# Data Exploration and Dataset Splitting (Dataset A)

In this notebook, we explore the processed peptide-HLA dataset and create a proper train/validation/test split.

The goal is to:
- understand the dataset structure
- check label distribution
- create a leakage-free split based on peptide sequences
- prepare Dataset A for model training

We ensure that the same peptide does not appear in both training and testing sets.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

In [2]:

df = pd.read_csv("../data/processed_training_data_with_features.csv")

df.head()

,index,peptide,HLA,Label,hla_sequence,Peptide_AF1,Peptide_AF2,Peptide_AF3,Peptide_AF4,Peptide_AF5,...,HLA_Z4,HLA_Z5,HLA_boman,HLA_hydrophobicity,HLA_charge,HLA_molecular_weight,HLA_aliphatic_index,HLA_instability_index,HLA_isoelectric_point,HLA_mz
0,0,KEHVFFSEY,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.173955,-0.346010,0.376536,-0.138244,-0.186528,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
1,1,DEGATLYRF,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.130313,-0.141878,0.624118,0.427633,0.340208,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
2,2,TLAARIKFL,HLA-A*02:01,0,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,-0.236566,-0.667240,0.421759,0.748899,0.552450,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
3,3,KETLNEYKQL,HLA-B*44:02,0,YYTKYREISTNTYENTAYIRYDDYTWAVDAYLSY,0.644379,-0.419675,0.461561,0.160180,0.170256,...,-0.595294,-0.155882,2.435294,-0.961765,-2.005081,4276.59504,51.764706,-11.114706,4.229102,2137.978906
4,4,STTDAEACY,HLA-A*01:01,0,YFAMYQENMAHTDANTLYIIYRDYTWVARVYRGY,-0.016626,-0.045450,-0.193642,0.402509,-0.348259,...,-0.101471,-0.141471,1.769118,-0.447059,0.085599,4259.78264,63.235294,-7.820588,7.501996,2129.501106


## Dataset Overview

We first inspect the dataset size, columns, and general structure.

In [3]:
print("Shape:", df.shape)
df.columns

Shape: (8969, 225)


Index(['index', 'peptide', 'HLA', 'Label', 'hla_sequence', 'Peptide_AF1',
       'Peptide_AF2', 'Peptide_AF3', 'Peptide_AF4', 'Peptide_AF5',
       ...
       'HLA_Z4', 'HLA_Z5', 'HLA_boman', 'HLA_hydrophobicity', 'HLA_charge',
       'HLA_molecular_weight', 'HLA_aliphatic_index', 'HLA_instability_index',
       'HLA_isoelectric_point', 'HLA_mz'],
      dtype='object', length=225)

## Label Distribution

We check how balanced the dataset is between immunogenic (1) and non-immunogenic (0) samples.

In [4]:
df["Label"].value_counts()

Label
0    4912
1    4057
Name: count, dtype: int64

In [5]:
df["Label"].value_counts(normalize=True)

Label
0    0.547664
1    0.452336
Name: proportion, dtype: float64

## Peptide Analysis

We examine how many unique peptides are present and whether duplicates exist.

In [6]:
print("Total samples:", len(df))
print("Unique peptides:", df["peptide"].nunique())

Total samples: 8969
Unique peptides: 8381


## Train/Test Split Strategy

To avoid data leakage, we split the dataset based on peptide sequences.

This ensures that:
- the same peptide does not appear in both training and testing sets
- the model does not memorize peptide-specific patterns

In [7]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["peptide"]
y = df["Label"]

gss = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(df, y, groups))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

Train size: (7157, 225)
Test size: (1812, 225)


## Leakage Check

We verify that no peptide appears in both training and testing sets.

In [8]:
train_peptides = set(train_df["peptide"])
test_peptides = set(test_df["peptide"])

overlap = train_peptides.intersection(test_peptides)
print("Number of overlapping peptides:", len(overlap))

Number of overlapping peptides: 0


## Validation Split

We further split the training data into training and validation sets using the same grouping strategy.

In [9]:
gss_val = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx2, val_idx = next(gss_val.split(train_df, train_df["Label"], groups=train_df["peptide"]))

train_final = train_df.iloc[train_idx2]
val_df = train_df.iloc[val_idx]

print("Final train size:", train_final.shape)
print("Validation size:", val_df.shape)

Final train size: (5718, 225)
Validation size: (1439, 225)


## Final Label Distribution

We check that the label distribution is similar across splits.

In [10]:
print("Train:")
print(train_final["Label"].value_counts(normalize=True))

print("\nValidation:")
print(val_df["Label"].value_counts(normalize=True))

print("\nTest:")
print(test_df["Label"].value_counts(normalize=True))

Train:
Label
0    0.54687
1    0.45313
Name: proportion, dtype: float64

Validation:
Label
0    0.539263
1    0.460737
Name: proportion, dtype: float64

Test:
Label
0    0.556843
1    0.443157
Name: proportion, dtype: float64


In [11]:
train_final.to_csv("../data/dataset_A_train.csv", index=False)
val_df.to_csv("../data/dataset_A_val.csv", index=False)
test_df.to_csv("../data/dataset_A_test.csv", index=False)

In [12]:
# save indices for later to use on other versions of datasets with different features
# in order to explore other feature methods (LLM embeddings or other packages to extract features)

np.save("../data/train_idx.npy", train_idx)
np.save("../data/test_idx.npy", test_idx)